# rembg 배경제거 PoC (이미지 생성 파트)

- **목적**: 사진 보존 모드의 1번 부품(배경 제거) 모델·옵션 비교
- **사용법**: 같은 폴더에 테스트 사진을 `테스트사진.jpg`로 넣고 위에서부터 실행
- **결론**: `isnet-general-use` + `post_process_mask=True`를 v1 설정으로 채택 (하단 실험 기록 참고)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from rembg import remove, new_session

In [ ]:
def show(im, size=380):
    """표시용 축소 미리보기 (원본 이미지는 건드리지 않음)"""
    p = im.copy()
    p.thumbnail((size, size))
    return p

In [ ]:
def compare(pairs, size=300):
    """[(제목, 이미지), ...] 를 한 줄로 나란히 비교 표시"""
    font = ImageFont.truetype("C:/Windows/Fonts/malgun.ttf", 18)
    gap, label_h = 12, 36
    W = len(pairs) * (size + gap) + gap
    canvas = Image.new("RGB", (W, size + label_h + gap * 2), "white")
    d = ImageDraw.Draw(canvas)
    for i, (title, im) in enumerate(pairs):
        p = im.copy()
        p.thumbnail((size, size))
        x = gap + i * (size + gap)
        tile = Image.new("RGBA", (size, size), "white")   # 투명 배경은 흰색 위에
        tile.paste(p, ((size - p.width) // 2, (size - p.height) // 2),
                   p if p.mode == "RGBA" else None)
        canvas.paste(tile.convert("RGB"), (x, gap))
        d.text((x + size // 2, size + gap + 16), title, font=font, fill="black", anchor="mm")
    return canvas

In [ ]:
# 이미지를 불러온다
img = Image.open("테스트사진.jpg")

In [ ]:
# u2net 기본
cut = remove(img)

In [ ]:
# 상위 모델인 isnet
sess = new_session("isnet-general-use")
cut2 = remove(img, session=sess)

In [ ]:
# isnet에 post_process_mask=True 옵션을 사용하여 마스크를 후처리하였을 때 더 깔끔한 결과를 얻는지 확인
cut3 = remove(img, session=sess, post_process_mask=True)

In [ ]:
compare([
    ("원본", img),
    ("u2net 기본", cut),
    ("isnet", cut2),
    ("isnet+후처리", cut3),
])

## 실험 기록 (2026-08-06)

테스트 사진: 오므라이스(접시·소스 포함) 1장

| 설정 | 결과 |
|---|---|
| u2net (기본) | 접시·소스가 부드럽게 일부 남음 — 음식 광고 용도로는 오히려 자연스러움 |
| isnet-general-use | 음식 위주로 또렷하게 분리. 단 검은 소스 일부 누락, 접시 제거로 공중부양 느낌 |
| isnet + post_process_mask | isnet 결과에서 가장자리 잔상이 정리됨 |

**관찰**

- 모델마다 "제품의 범위"를 다르게 해석함 — u2net은 접시까지, isnet은 음식만
- 광고 관점에서는 "깔끔한 분리"가 항상 정답이 아님 (접시 없는 음식은 어색할 수 있음)
- → **"접시는 제품인가 배경인가"** 기준을 먼저 정의해야 판정 가능

**결론**

사진 1장으로는 승자를 확정할 수 없음 → 업종·장면별 여러 사진(골든셋)으로 비교 평가 예정

**추후 실험 과제**

- 골든셋 구성 후 모델별 일괄 비교
- 접시 포함 마스크 옵션 검토
- 마스크 품질 자동 판정 기준 만들기